In [41]:
import os
import kagglehub
import pandas as pd
from pathlib import Path


from sklearn.model_selection import train_test_split

Chargement du dataset

In [42]:
# Download latest version
path = kagglehub.dataset_download("algozee/financial-transaction-fraud-dataset")

print("Path to dataset files:", path)

df = pd.read_csv(os.path.join(path, "FraudShield_Banking_Data.csv"))
print(f'Number of rows: {df.shape[0]} | Number of columns:{df.shape[1]}')
print(f'Fraud Label : {df['Fraud_Label'].value_counts(normalize=True)}')

Path to dataset files: C:\Users\tapon\.cache\kagglehub\datasets\algozee\financial-transaction-fraud-dataset\versions\1
Number of rows: 50000 | Number of columns:25
Fraud Label : Fraud_Label
Normal    0.951536
Fraud     0.048464
Name: proportion, dtype: float64


Convertion Yes/No -> 1/0

In [43]:
binary_cols = [
    'Is_International_Transaction',
    'Is_New_Merchant',
    'Unusual_Time_Transaction'
]

for col in binary_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({'yes': 1, 'no': 0})
    )
for col in binary_cols:
    df[col] = df[col].astype("Int64")
print(df[['Is_International_Transaction', 'Is_New_Merchant', 'Unusual_Time_Transaction']])

       Is_International_Transaction  Is_New_Merchant  Unusual_Time_Transaction
0                                 1                1                         0
1                                 1                1                         0
2                                 1                0                         1
3                                 0                1                         1
4                                 0                1                         0
...                             ...              ...                       ...
49995                             1                0                         0
49996                             1                1                         0
49997                             1                1                         0
49998                             1                0                         0
49999                             0                1                         1

[50000 rows x 3 columns]


Gestion de l'heure

In [44]:
from copy import error
df['Transaction_Time'] = pd.to_datetime(df['Transaction_Time'], format = '%H:%M', errors = 'coerce')
df['Hour'] = df['Transaction_Time'].dt.hour
df['Minute'] = df['Transaction_Time'].dt.minute

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format = '%Y-%m-%d', errors = 'coerce')
df['Year'] = df['Transaction_Date'].dt.year
df['Month'] = df['Transaction_Date'].dt.month
df['Day'] = df['Transaction_Date'].dt.day

Imputation des variables maquantes numeric par la moyenne et les variables categorielles par le mode

In [45]:
# Récupérer les noms des colonnes numériques standard (float64, numpy int64)
# Ces colonnes peuvent être imputées par la moyenne.
standard_numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Récupérer les noms des colonnes de type 'Int64' (entier nullable de pandas)
# Ces colonnes sont souvent des drapeaux binaires et sont mieux imputées par le mode.
nullable_int_cols = df.select_dtypes(include=['Int64']).columns

# S'assurer qu'aucune colonne Int64 n'est incluse par inadvertance dans standard_numeric_cols
# Cette étape rend l'imputation plus robuste.
standard_numeric_cols = standard_numeric_cols.drop(nullable_int_cols, errors='ignore')

if not standard_numeric_cols.empty:
    df[standard_numeric_cols] = df[standard_numeric_cols].fillna(df[standard_numeric_cols].mean())

# Imputer les valeurs manquantes des colonnes de type 'Int64' par le mode.
if not nullable_int_cols.empty:
    for col in nullable_int_cols:
        if df[col].isnull().any():
            if not df[col].mode().empty:
                df[col] = df[col].fillna(df[col].mode()[0])

# Récupérer les noms des colonnes catégorielles (object)
# Ces colonnes sont imputées par le mode.
object_cols = df.select_dtypes(include=['object']).columns
if not object_cols.empty:
    for col in object_cols:
        if df[col].isnull().any():
            if not df[col].mode().empty:
                df[col] = df[col].fillna(df[col].mode()[0])

Vérification des veleurs nulles

In [46]:
df.isnull().sum().sort_values(ascending=False)

Transaction_Time                         9
Transaction_Date                         3
Customer_ID                              0
Transaction_ID                           0
Transaction_Amount (in Million)          0
Transaction_Type                         0
Merchant_ID                              0
Merchant_Category                        0
Transaction_Location                     0
Customer_Home_Location                   0
Distance_From_Home                       0
Device_ID                                0
IP_Address                               0
Card_Type                                0
Account_Balance (in Million)             0
Daily_Transaction_Count                  0
Weekly_Transaction_Count                 0
Avg_Transaction_Amount (in Million)      0
Max_Transaction_Last_24h (in Million)    0
Is_International_Transaction             0
Is_New_Merchant                          0
Failed_Transaction_Count                 0
Unusual_Time_Transaction                 0
Previous_Fr

Véfication des valeurs doubles

In [47]:
number_duplicate = df.duplicated().sum()
print(f'Number of duplicate rows: {number_duplicate}')

Number of duplicate rows: 0


Recuperer une copie de la base de données


In [48]:
clean_df = df.copy()

Suppression des colonnes

In [49]:
clean_df = clean_df.drop(columns=['Transaction_ID', 'Customer_ID', 'Transaction_Time', 'Transaction_Date', 'Merchant_ID', 'Device_ID', 'IP_Address'])

Transformation des variables catégorielles en utilisant One-Hot Encodin

In [50]:
categorical_cols = [
    'Transaction_Type',
    'Merchant_Category',
    'Transaction_Location',
    'Customer_Home_Location',
    'Card_Type'
]
clean_df = pd.get_dummies(
    clean_df,
    columns=categorical_cols,
    dtype=int
)
print(clean_df)



       Transaction_Amount (in Million)  Distance_From_Home  \
0                                  6.0               466.0   
1                                  9.0               215.0   
2                                  3.0               216.0   
3                                  1.0               408.0   
4                                  1.0               209.0   
...                                ...                 ...   
49995                              4.0               571.0   
49996                              4.0               433.0   
49997                              9.0               161.0   
49998                              1.0               317.0   
49999                              7.0               494.0   

       Account_Balance (in Million)  Daily_Transaction_Count  \
0                              30.0                      4.0   
1                               4.0                      4.0   
2                              38.0                      5.0   

Extraction de la variable cible et conversion des valeurs en numeriques : Normal/Fraud -> 1/0

In [51]:
target_df = clean_df['Fraud_Label']
# Handle NaN values before converting to integer by filling with the mode
target_df = target_df.fillna(target_df.mode()[0])
print('conversion des valeurs en numeriques Normal/Fraud -> 1/0')
target_df = target_df.map({'Normal': 0, 'Fraud': 1})
print('convert target_df en integer')
target_df = target_df.astype(int)
target_df.head(5)

conversion des valeurs en numeriques Normal/Fraud -> 1/0
convert target_df en integer


0    0
1    0
2    0
3    0
4    0
Name: Fraud_Label, dtype: int64

Séparation des données

In [52]:
# X : variables explicatives
# y : variable cible

X = clean_df.drop("Fraud_Label", axis=1)
y = target_df

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Sauvegarde des données

In [62]:
print("Sauvegarde des données dans un repertoire local")

Path('../clean_data').mkdir(parents=True, exist_ok=True)
X_train.to_csv("../clean_data/X_train.csv", index=False, encoding='utf-8')
X_test.to_csv("../clean_data/X_test.csv", index=False, encoding='utf-8')
y_test.to_csv("../clean_data/y_test.csv", index=False, encoding='utf-8')
y_train.to_csv("../clean_data/y_train.csv", index=False, encoding='utf-8')

Sauvegarde des données dans un repertoire local
